In [ ]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb prophet

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

In [ ]:
import logging

import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
import wandb
from prophet import Prophet
from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline

logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.WARNING)

In [3]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
train.shape

(421570, 5)

In [4]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [5]:
HOLIDAYS = pd.DataFrame({
    'holiday': (['super_bowl'] * 4 + ['labor_day'] * 4 +
                ['thanksgiving'] * 4 + ['christmas'] * 4 + ['pre_christmas'] * 4),
    'ds': pd.to_datetime([
        '2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08',
        '2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06',
        '2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29',
        '2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27',
        '2010-12-24', '2011-12-23', '2012-12-21', '2013-12-20',
    ]),
})

In [6]:
class ProphetForecaster(BaseEstimator):
    def __init__(self, holidays=None, yearly_seasonality=True, changepoint_prior_scale=0.05):
        self.holidays = holidays
        self.yearly_seasonality = yearly_seasonality
        self.changepoint_prior_scale = changepoint_prior_scale

    def fit(self, X, y):
        df = X[['Store', 'Dept', 'Date']].copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df['y'] = y.values if hasattr(y, 'values') else y

        totals = df.groupby(['Store', 'Date'])['y'].sum().reset_index()

        self.models_ = {}
        for store, g in totals.groupby('Store'):
            m = Prophet(
                yearly_seasonality=self.yearly_seasonality,
                weekly_seasonality=False,
                daily_seasonality=False,
                holidays=self.holidays,
                changepoint_prior_scale=self.changepoint_prior_scale,
            )
            m.fit(g.rename(columns={'Date': 'ds'})[['ds', 'y']])
            self.models_[store] = m

        df['woy'] = df['Date'].dt.isocalendar().week.astype(int)
        df = df.merge(totals.rename(columns={'y': 'total'}), on=['Store', 'Date'])
        df['share'] = np.where(df['total'] != 0, df['y'] / df['total'], 0.0)

        self.woy_share_ = df.groupby(['Store', 'Dept', 'woy'])['share'].mean().to_dict()
        self.pair_share_ = df.groupby(['Store', 'Dept'])['share'].mean().to_dict()
        return self

    def predict(self, X):
        df = X.copy().reset_index(drop=True)
        df['Date'] = pd.to_datetime(df['Date'])
        df['woy'] = df['Date'].dt.isocalendar().week.astype(int)

        preds = np.zeros(len(df))
        for store, g in df.groupby('Store'):
            model = self.models_.get(store)
            if model is None:
                continue
            future = pd.DataFrame({'ds': sorted(g['Date'].unique())})
            forecast = model.predict(future).set_index('ds')['yhat']
            totals = g['Date'].map(forecast).values

            shares = np.array([
                self.woy_share_.get(
                    (store, dept, woy),
                    self.pair_share_.get((store, dept), 0.0),
                )
                for dept, woy in zip(g['Dept'], g['woy'])
            ])
            preds[g.index] = totals * shares

        return np.clip(preds, 0, None)

In [ ]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('Prophet_Training')

In [ ]:
with mlflow.start_run(run_name='Prophet_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='Prophet_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    wandb.log({'negative_sales_rows': n_negative})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

In [8]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

[(np.datetime64('2010-10-01T00:00:00.000000000'),
  np.datetime64('2011-06-03T00:00:00.000000000')),
 (np.datetime64('2011-06-03T00:00:00.000000000'),
  np.datetime64('2012-02-03T00:00:00.000000000')),
 (np.datetime64('2012-02-03T00:00:00.000000000'),
  np.datetime64('2012-10-05T00:00:00.000000000'))]

In [ ]:
params = dict(yearly_seasonality=True, changepoint_prior_scale=0.05)

with mlflow.start_run(run_name='Prophet_CV'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='Prophet_CV', config=params, reinit='finish_previous')

    fold_scores = []
    for i, (train_end, val_end) in enumerate(splits):
        train_mask = train['Date'] <= train_end
        val_mask = (train['Date'] > train_end) & (train['Date'] <= val_end)

        model = ProphetForecaster(holidays=HOLIDAYS, **params)
        model.fit(train.loc[train_mask, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[train_mask])

        preds = model.predict(train.loc[val_mask, ['Store', 'Dept', 'Date', 'IsHoliday']])
        score = wmae(y_train[val_mask].values, preds, train.loc[val_mask, 'IsHoliday'].values)
        fold_scores.append(score)
        mlflow.log_metric('wmae', score, step=i)
        wandb.log({'fold': i, 'wmae': score})

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores))})
    wandb.finish()

fold_scores

In [ ]:
with mlflow.start_run(run_name='Prophet_Final'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='Prophet_Final', config=params, reinit='finish_previous')

    pipeline = Pipeline([
        ('model', ProphetForecaster(holidays=HOLIDAYS, **params)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train)

    preds = pipeline.predict(train[['Store', 'Dept', 'Date', 'IsHoliday']])
    train_wmae = wmae(y_train.values, preds, train['IsHoliday'].values)
    mlflow.log_metric('train_wmae', train_wmae)
    wandb.log({'train_wmae': train_wmae})

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')

    joblib.dump(pipeline, 'prophet_pipeline.joblib')
    artifact = wandb.Artifact('prophet_pipeline', type='model')
    artifact.add_file('prophet_pipeline.joblib')
    wandb.log_artifact(artifact)
    wandb.finish()

train_wmae

In [ ]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_Prophet.ipynb in Colab"
    !git push